In [2]:
# etl_UrbanPopulation.py

from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
from pandera import Column, Check, DataFrameSchema

API_URL = "https://api.worldbank.org/v2/country/all/indicator/SP.URB.TOTL.IN.ZS"

# ==========================
# Extracción
# ==========================
@task
def extract_urban_population(year_from: int = 2000, year_to: int = 2023, per_page: int = 1000):
    all_data = []
    page = 1
    total_pages = None

    while True:
        url = f"{API_URL}?format=json&date={year_from}:{year_to}&per_page={per_page}&page={page}"
        response = requests.get(url)
        response.raise_for_status()
        json_data = response.json()

        if total_pages is None:
            total_pages = json_data[0]["pages"]

        data_page = json_data[1]
        all_data.extend(data_page)

        print(f"🔄 Descargando página {page} de {total_pages} ({len(all_data)} filas acumuladas)")

        if page >= total_pages:
            break
        page += 1

    return all_data

# ==========================
# Transformación
# ==========================
@task
def transform_urban_population(data):
    df = pd.DataFrame(data)

    # Seleccionar columnas relevantes
    df_clean = df[["countryiso3code", "country", "date", "value"]].copy()

    # Extraer nombre del país
    df_clean["country_name"] = df_clean["country"].apply(
        lambda x: x.get("value") if isinstance(x, dict) else None
    )

    # Renombrar columnas
    df_clean = df_clean.rename(columns={
        "countryiso3code": "id_country",
        "date": "year",
        "value": "urban_population_pct"
    })[["id_country", "country_name", "year", "urban_population_pct"]]

    # Convertir año a int
    df_clean["year"] = df_clean["year"].astype(int)

    # 🔑 Filtrar: solo países con id_country de 3 letras
    df_clean = df_clean[df_clean["id_country"].str.len() == 3]

    return df_clean

# ==========================
# Validación
# ==========================
urban_schema = DataFrameSchema({
    "id_country": Column(str, [
        Check.str_length(3, 3)
    ]),
    "country_name": Column(str, nullable=True),
    "year": Column(int, [
        Check.ge(1900),
        Check.le(2100)
    ]),
    "urban_population_pct": Column(float, nullable=True, checks=[
        Check.ge(0),
        Check.le(100)
    ]),
})

@task
def validate_urban_population(df: pd.DataFrame) -> pd.DataFrame:
    return urban_schema.validate(df)

# ==========================
# Carga
# ==========================
@task
def load_urban_population(df: pd.DataFrame):
    file_path = Path("../stage/country_urban_population.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"💾 Guardado {len(df)} filas en {file_path}")

# ==========================
# Flow
# ==========================
@flow(name="etl_urban_population_flow")
def etl_urban_population(year_from: int = 2000, year_to: int = 2023):
    data = extract_urban_population(year_from, year_to)
    df = transform_urban_population(data)
    df = validate_urban_population(df)
    load_urban_population(df)

if __name__ == "__main__":
    etl_urban_population(year_from=2000, year_to=2025)


05:53:44.827 | INFO    | Flow run 'nostalgic-stoat' - Beginning flow run 'nostalgic-stoat' for flow 'etl_urban_population_flow'

🔄 Descargando página 1 de 7 (1000 filas acumuladas)
🔄 Descargando página 2 de 7 (2000 filas acumuladas)
🔄 Descargando página 3 de 7 (3000 filas acumuladas)
🔄 Descargando página 4 de 7 (4000 filas acumuladas)
🔄 Descargando página 5 de 7 (5000 filas acumuladas)
🔄 Descargando página 6 de 7 (6000 filas acumuladas)
🔄 Descargando página 7 de 7 (6384 filas acumuladas)


05:53:49.809 | INFO    | Task run 'extract_urban_population-989' - Finished in state Completed()

05:53:50.856 | INFO    | Task run 'transform_urban_population-79f' - Finished in state Completed()

05:53:51.082 | INFO    | Task run 'validate_urban_population-dfa' - Finished in state Completed()

💾 Guardado 6264 filas en ..\stage\country_urban_population.csv


05:53:51.309 | INFO    | Task run 'load_urban_population-6cd' - Finished in state Completed()

05:53:51.322 | INFO    | Flow run 'nostalgic-stoat' - Finished in state Completed()